# Stage 1 Reduced Formula Fit

This notebook fits the reduced theoretical formulas from `THEORETICAL_MODEL.md` to Stage 1 numerical analysis CSV files. It is self-contained and can run in Colab if the CSV files are uploaded.

## 1. Imports And Input Paths

In [ ]:
# Uncomment only if pandas or matplotlib is missing.
# %pip install pandas matplotlib

import csv
import math
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

print('pandas_available =', pd is not None)
print('matplotlib_available =', plt is not None)

In [ ]:
# Colab upload path. Upload attention_length_summary.csv and controlled_examples.csv.
try:
    from google.colab import files
    print('Colab detected. Upload attention_length_summary.csv and controlled_examples.csv.')
    uploaded = files.upload()
except Exception:
    uploaded = {}

# For local repo runs, this path matches the CLI numerical analysis output.
ANALYSIS_DIR = Path('runs/stage1_transformer_maxpool/numerical_analysis')

# For Colab uploads, use current directory if the default analysis directory is absent.
if not (ANALYSIS_DIR / 'attention_length_summary.csv').exists():
    ANALYSIS_DIR = Path('.')

OUTPUT_DIR = Path('theoretical_formula_fit_notebook')
ATTENTION_QUERY = 'mean_over_queries'

print('ANALYSIS_DIR =', ANALYSIS_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

## 2. Fit Helpers

In [ ]:
def read_csv(path):
    if not Path(path).exists():
        raise FileNotFoundError(f'missing input CSV: {path}')
    with Path(path).open('r', newline='', encoding='utf-8') as file:
        return list(csv.DictReader(file))


def write_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text('', encoding='utf-8')
        return
    fieldnames = list(rows[0])
    for row in rows[1:]:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def target_attention_mass(sequence_length, target_score_margin):
    return 1.0 / (1.0 + (sequence_length - 1.0) * math.exp(-target_score_margin))


def average(values):
    return sum(values) / len(values)


def regression_metrics(observed, predicted):
    residuals = [actual - estimate for actual, estimate in zip(observed, predicted)]
    squared_errors = [value * value for value in residuals]
    absolute_errors = [abs(value) for value in residuals]
    observed_mean = average(observed)
    total_sum_of_squares = sum((value - observed_mean) ** 2 for value in observed)
    residual_sum_of_squares = sum(squared_errors)
    return {
        'mean_absolute_error': average(absolute_errors),
        'root_mean_squared_error': math.sqrt(average(squared_errors)),
        'residual_sum_of_squares': residual_sum_of_squares,
        'r_squared': 1.0 - residual_sum_of_squares / total_sum_of_squares if total_sum_of_squares > 0 else float('nan'),
    }


def fit_attention_margin(lengths, observed_attention):
    def objective(target_score_margin):
        predicted = [target_attention_mass(length, target_score_margin) for length in lengths]
        return sum((observed - estimate) ** 2 for observed, estimate in zip(observed_attention, predicted))

    lower, upper = -20.0, 20.0
    golden_ratio_conjugate = (math.sqrt(5.0) - 1.0) / 2.0
    left = upper - golden_ratio_conjugate * (upper - lower)
    right = lower + golden_ratio_conjugate * (upper - lower)
    left_value = objective(left)
    right_value = objective(right)
    for _ in range(200):
        if left_value > right_value:
            lower = left
            left = right
            left_value = right_value
            right = lower + golden_ratio_conjugate * (upper - lower)
            right_value = objective(right)
        else:
            upper = right
            right = left
            right_value = left_value
            left = upper - golden_ratio_conjugate * (upper - lower)
            left_value = objective(left)
    return (lower + upper) / 2.0


In [ ]:
def solve_linear_system(matrix, vector):
    size = len(vector)
    augmented = [row[:] + [value] for row, value in zip(matrix, vector)]
    for pivot_index in range(size):
        best_row = max(range(pivot_index, size), key=lambda row_index: abs(augmented[row_index][pivot_index]))
        if abs(augmented[best_row][pivot_index]) < 1e-12:
            raise ValueError('linear system is singular')
        if best_row != pivot_index:
            augmented[pivot_index], augmented[best_row] = augmented[best_row], augmented[pivot_index]
        pivot = augmented[pivot_index][pivot_index]
        for column_index in range(pivot_index, size + 1):
            augmented[pivot_index][column_index] /= pivot
        for row_index in range(size):
            if row_index == pivot_index:
                continue
            factor = augmented[row_index][pivot_index]
            for column_index in range(pivot_index, size + 1):
                augmented[row_index][column_index] -= factor * augmented[pivot_index][column_index]
    return [augmented[row_index][size] for row_index in range(size)]


def fit_linear_model(features, targets):
    feature_count = len(features[0])
    normal_matrix = [
        [sum(row[left] * row[right] for row in features) for right in range(feature_count)]
        for left in range(feature_count)
    ]
    normal_vector = [sum(row[index] * target for row, target in zip(features, targets)) for index in range(feature_count)]
    return solve_linear_system(normal_matrix, normal_vector)


def predict_linear_model(features, coefficients):
    return [sum(value * coefficient for value, coefficient in zip(row, coefficients)) for row in features]


def log_growth(sequence_length):
    return math.log(sequence_length)


def sqrt_log_growth(sequence_length):
    return math.sqrt(2.0 * math.log(sequence_length))

## 3. Load Stage 1 Numerical Analysis Data

In [ ]:
attention_rows_raw = read_csv(ANALYSIS_DIR / 'attention_length_summary.csv')
controlled_rows_raw = read_csv(ANALYSIS_DIR / 'controlled_examples.csv')

attention_by_length = {}
for row in attention_rows_raw:
    if row.get('query') == ATTENTION_QUERY:
        attention_by_length[int(row['length'])] = float(row['target_attention_mean'])

positive_logit_by_length = {}
for row in controlled_rows_raw:
    if row.get('label_type') == 'positive':
        positive_logit_by_length[int(row['length'])] = float(row['logit'])

shared_lengths = sorted(set(attention_by_length) & set(positive_logit_by_length))
if len(shared_lengths) < 3:
    raise ValueError('Need at least three shared lengths.')

lengths = [float(length) for length in shared_lengths]
observed_attention = [attention_by_length[length] for length in shared_lengths]
observed_logits = [positive_logit_by_length[length] for length in shared_lengths]

loaded_rows = [
    {'length': int(length), 'observed_attention': attention, 'observed_logit': logit}
    for length, attention, logit in zip(lengths, observed_attention, observed_logits)
]
pd.DataFrame(loaded_rows) if pd is not None else loaded_rows

## 4. Fit Target Attention Formula

In [ ]:
target_score_margin = fit_attention_margin(lengths, observed_attention)
fitted_attention = [target_attention_mass(length, target_score_margin) for length in lengths]
attention_metrics = regression_metrics(observed_attention, fitted_attention)

attention_fit_rows = [
    {
        'length': int(length),
        'observed_attention': observed,
        'predicted_attention': predicted,
        'residual': observed - predicted,
        'target_score_margin': target_score_margin,
    }
    for length, observed, predicted in zip(lengths, observed_attention, fitted_attention)
]

attention_summary_rows = [{'target_score_margin': target_score_margin, **attention_metrics}]
print('target_score_margin =', target_score_margin)
pd.DataFrame(attention_fit_rows) if pd is not None else attention_fit_rows

## 5. Fit Final Logit Formula Candidates

In [ ]:
model_specs = [
    ('observed_attention_only', observed_attention, 'none', None),
    ('fitted_attention_only', fitted_attention, 'none', None),
    ('observed_attention_plus_log_length', observed_attention, 'log_length', log_growth),
    ('fitted_attention_plus_log_length', fitted_attention, 'log_length', log_growth),
    ('observed_attention_plus_sqrt_2_log_length', observed_attention, 'sqrt_2_log_length', sqrt_log_growth),
    ('fitted_attention_plus_sqrt_2_log_length', fitted_attention, 'sqrt_2_log_length', sqrt_log_growth),
]

logit_summary_rows = []
logit_fit_rows = []

for model_name, attention_values, growth_name, growth_function in model_specs:
    if growth_function is None:
        growth_values = [0.0 for _ in lengths]
        features = [[1.0, attention] for attention in attention_values]
        coefficients = fit_linear_model(features, observed_logits)
        classifier_bias, target_signal_strength = coefficients
        non_target_interference_strength = 0.0
    else:
        growth_values = [growth_function(length) for length in lengths]
        features = [[1.0, attention, -growth] for attention, growth in zip(attention_values, growth_values)]
        coefficients = fit_linear_model(features, observed_logits)
        classifier_bias, target_signal_strength, non_target_interference_strength = coefficients

    predicted_logits = predict_linear_model(features, coefficients)
    metrics = regression_metrics(observed_logits, predicted_logits)
    logit_summary_rows.append({
        'model_name': model_name,
        'attention_source': 'observed' if model_name.startswith('observed') else 'fitted',
        'growth_name': growth_name,
        'target_score_margin': target_score_margin,
        'classifier_bias': classifier_bias,
        'target_signal_strength': target_signal_strength,
        'non_target_interference_strength': non_target_interference_strength,
        **metrics,
    })
    for length, observed_attention_value, fitted_attention_value, attention_value, growth_value, observed_logit, predicted_logit in zip(lengths, observed_attention, fitted_attention, attention_values, growth_values, observed_logits, predicted_logits):
        logit_fit_rows.append({
            'model_name': model_name,
            'length': int(length),
            'observed_attention': observed_attention_value,
            'fitted_attention': fitted_attention_value,
            'attention_used_by_model': attention_value,
            'interference_growth': growth_value,
            'observed_logit': observed_logit,
            'predicted_logit': predicted_logit,
            'residual': observed_logit - predicted_logit,
        })

summary_sorted = sorted(logit_summary_rows, key=lambda row: row['root_mean_squared_error'])
pd.DataFrame(summary_sorted) if pd is not None else summary_sorted

## 6. Save CSV Outputs

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_csv(OUTPUT_DIR / 'attention_fit_summary.csv', attention_summary_rows)
write_csv(OUTPUT_DIR / 'attention_fit_by_length.csv', attention_fit_rows)
write_csv(OUTPUT_DIR / 'logit_fit_summary.csv', logit_summary_rows)
write_csv(OUTPUT_DIR / 'logit_fit_by_length.csv', logit_fit_rows)
print('Saved fit CSV files to', OUTPUT_DIR)

## 7. Plot Fits

In [ ]:
if plt is None:
    print('matplotlib is unavailable; skipping plots.')
else:
    figure_dir = OUTPUT_DIR / 'figures'
    figure_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(7, 4.2), dpi=160)
    ax.plot(shared_lengths, observed_attention, marker='o', label='observed')
    ax.plot(shared_lengths, fitted_attention, marker='o', label='fitted formula')
    ax.set_xscale('log')
    ax.set_xlabel('sequence length')
    ax.set_ylabel('target attention mass')
    ax.set_title('Target attention formula fit')
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(figure_dir / 'target_attention_fit.png', bbox_inches='tight')
    plt.show()

    best_model_name = summary_sorted[0]['model_name']
    best_rows = [row for row in logit_fit_rows if row['model_name'] == best_model_name]
    best_rows = sorted(best_rows, key=lambda row: row['length'])
    fig, ax = plt.subplots(figsize=(7, 4.2), dpi=160)
    ax.plot([row['length'] for row in best_rows], [row['observed_logit'] for row in best_rows], marker='o', label='observed')
    ax.plot([row['length'] for row in best_rows], [row['predicted_logit'] for row in best_rows], marker='o', label='fitted formula')
    ax.axhline(0.0, color='black', linestyle='--', linewidth=1)
    ax.set_xscale('log')
    ax.set_xlabel('sequence length')
    ax.set_ylabel('positive logit')
    ax.set_title(f'Best logit formula fit: {best_model_name}')
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(figure_dir / 'positive_logit_fit.png', bbox_inches='tight')
    plt.show()
    print('Saved figures to', figure_dir)